# Lumpy 03: Results Review

Notebook version: 3

Reads the saved lumpy outputs and keeps the review pointed at the useful lenses:

- preferred demand-planning view: 18-month window, 3-month lag, 3-month rolling WMAPE
- allocation check: SKU quantity across the forecast horizon
- diagnostic only: exact SKU-month WMAPE

This notebook does not train models.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    pio.templates.default = "plotly_white"
except ImportError:
    go = None
    pio = None

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


NOTEBOOK_VERSION = 3
REQUIRED_WINDOW_MONTHS = 18
REQUIRED_LAG_MONTHS = 3


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [
        start,
        start.parent,
        start.parent.parent,
        start / "Inchscape Repo",
        start.parent / "Inchscape Repo",
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "results").exists():
            return candidate.resolve()
    raise FileNotFoundError("Open this notebook from the Inchscape Repo folder or update find_repo_root().")


def read_table(filename, *, parse_dates=None, required=True):
    path = table_dir / filename
    if path.exists():
        return pd.read_csv(path, parse_dates=parse_dates)
    if required:
        raise FileNotFoundError(f"Missing required lumpy output: {path}")
    return pd.DataFrame()


root = find_repo_root()
table_dir = root / "results" / "lumpy_outputs" / "tables"

metric_suite = read_table("lumpy_metric_suite_model_summary.csv")
monthly_total_model_summary = read_table("lumpy_monthly_total_model_summary.csv")
sku_horizon_model_summary = read_table("lumpy_sku_horizon_model_summary.csv")
monthly_totals = read_table("lumpy_monthly_total_results.csv", parse_dates=["month"], required=False)
lag_comparison_report = read_table("lumpy_agent_lag_comparison_report.csv", required=False)
metric_decision_report = read_table("lumpy_agent_metric_decision_report.csv", required=False)

required_metric = metric_suite.loc[
    metric_suite["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & metric_suite["gap_months"].eq(REQUIRED_LAG_MONTHS)
].copy()

monthly_required = monthly_total_model_summary.loc[
    monthly_total_model_summary["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & monthly_total_model_summary["gap_months"].eq(REQUIRED_LAG_MONTHS)
].copy()

sku_horizon_required = sku_horizon_model_summary.loc[
    sku_horizon_model_summary["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & sku_horizon_model_summary["gap_months"].eq(REQUIRED_LAG_MONTHS)
].copy()

if required_metric.empty:
    raise ValueError("No 18-month window / 3-month lag rows found. Re-run Lumpy 02 with that benchmark enabled.")

required_scorecard = required_metric.sort_values(["phase1_decision_score_percent", "model"]).copy()
lead_model = required_scorecard.iloc[0]["model"]

overall_best_by_model = (
    metric_suite.sort_values(["phase1_decision_score_percent", "model"])
    .drop_duplicates("model", keep="first")
    .reset_index(drop=True)
)
next_best_models = [model for model in overall_best_by_model["model"].tolist() if model != lead_model][:3]
plot_models = [lead_model] + next_best_models

print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Project root: {root}")
print(f"Required benchmark: {REQUIRED_WINDOW_MONTHS}m window / {REQUIRED_LAG_MONTHS}m lag")
print(f"Lead model on required benchmark: {lead_model}")
print(f"Plot models: {plot_models}")
if go is None:
    print("Plotly is not installed in this kernel. Install requirements.txt or run: python -m pip install plotly")


## Scorecard


In [ ]:
score_columns = [
    "window_label",
    "model",
    "phase1_decision_score_percent",
    "monthly_total_wmape_percent",
    "horizon_sku_wmape_percent",
    "sku_month_wmape_percent",
]
display(required_scorecard[score_columns])

phase2_action = required_scorecard[score_columns].copy()
phase2_action["phase2_action"] = np.select(
    [
        phase2_action["model"].eq(lead_model),
        phase2_action["model"].isin(["Aggregate Allocation", "Hurdle Random Forest"]),
        phase2_action["model"].eq("Zero Forecast"),
    ],
    [
        "keep_as_baseline_to_beat",
        "pause_default_runtime",
        "control_only",
    ],
    default="cheap_comparator",
)
display(phase2_action)

display(
    overall_best_by_model[
        [
            "model",
            "window_label",
            "phase1_decision_score_percent",
            "monthly_total_wmape_percent",
            "horizon_sku_wmape_percent",
        ]
    ].head(8)
)


## Plotly Forecast: 18-Month Window / 3-Month Lag


In [ ]:
if go is None:
    print("Plotly is not installed, so the forecast chart is skipped.")
elif monthly_totals.empty:
    print("No monthly-total detail table found, so the forecast chart is skipped.")
else:
    required_monthly = monthly_totals.loc[
        monthly_totals["test_months"].eq(REQUIRED_WINDOW_MONTHS)
        & monthly_totals["gap_months"].eq(REQUIRED_LAG_MONTHS)
    ].copy()
    latest_fold = required_monthly["global_fold_id"].max() if "global_fold_id" in required_monthly.columns else required_monthly["fold_id"].max()
    fold_column = "global_fold_id" if "global_fold_id" in required_monthly.columns else "fold_id"
    latest = required_monthly.loc[required_monthly[fold_column].eq(latest_fold)].copy()
    latest = latest.loc[latest["model"].isin(plot_models)].copy()

    actual = latest.groupby("month", as_index=False)["actual"].first().sort_values("month")

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=actual["month"],
            y=actual["actual"],
            mode="lines+markers",
            name="Actual",
            line=dict(color="#111827", width=4),
        )
    )

    colors = ["#2563eb", "#16a34a", "#dc2626", "#9333ea"]
    for model_index, model_name in enumerate(plot_models):
        model_data = latest.loc[latest["model"].eq(model_name)].sort_values("month")
        if model_data.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=model_data["month"],
                y=model_data["forecast"],
                mode="lines+markers",
                name=model_name,
                line=dict(
                    color=colors[model_index % len(colors)],
                    width=4 if model_name == lead_model else 2,
                    dash="solid" if model_name == lead_model else "dash",
                ),
            )
        )

    fig.update_layout(
        title=f"Latest Fold Forecast - {REQUIRED_WINDOW_MONTHS}m Window / {REQUIRED_LAG_MONTHS}m Lag",
        xaxis_title="Month",
        yaxis_title="Lumpy demand",
        legend_title="Series",
        hovermode="x unified",
        height=560,
        margin=dict(l=60, r=30, t=80, b=60),
    )
    fig.show()


## WMAPE Bar: Required Benchmark


In [ ]:
wmape_bar_data = monthly_required.sort_values(["mean_rolling_3m_wmape_percent", "model"]).copy()
wmape_columns = [
    "window_label",
    "model",
    "mean_rolling_3m_wmape_percent",
    "monthly_total_wmape_percent",
    "mean_monthly_wmape_percent",
    "actual_total",
    "forecast_total",
]
display(wmape_bar_data[wmape_columns])

if go is None:
    print("Plotly is not installed, so the WMAPE bar chart is skipped.")
elif wmape_bar_data.empty:
    print("No monthly-total summary rows found for the required benchmark.")
else:
    bar_colors = np.where(
        wmape_bar_data["model"].eq(lead_model),
        "#2563eb",
        np.where(wmape_bar_data["model"].isin(plot_models), "#16a34a", "#9ca3af"),
    )
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=wmape_bar_data["model"],
            y=wmape_bar_data["mean_rolling_3m_wmape_percent"],
            marker_color=bar_colors,
            text=wmape_bar_data["mean_rolling_3m_wmape_percent"].round(1).astype(str) + "%",
            textposition="outside",
            name="3-month rolling WMAPE",
        )
    )
    fig.add_hline(
        y=40,
        line_dash="dash",
        line_color="#16a34a",
        annotation_text="40% strong marker",
        annotation_position="top left",
    )
    fig.add_hline(
        y=70,
        line_dash="dash",
        line_color="#dc2626",
        annotation_text="70% usable marker",
        annotation_position="top left",
    )
    fig.update_layout(
        title=f"3-Month Rolling WMAPE - {REQUIRED_WINDOW_MONTHS}m Window / {REQUIRED_LAG_MONTHS}m Lag",
        xaxis_title="Model",
        yaxis_title="Mean rolling 3-month WMAPE %",
        yaxis=dict(range=[0, max(80, float(wmape_bar_data["mean_rolling_3m_wmape_percent"].max()) * 1.18)]),
        height=560,
        margin=dict(l=60, r=30, t=80, b=110),
    )
    fig.show()


## SKU-Level Forecast Examples


In [ ]:
sku_forecast_path = table_dir / "lumpy_backtest_forecasts.csv"
sku_forecast_columns = [
    "sku_id",
    "month",
    "demand",
    "forecast",
    "model",
    "fold_id",
    "global_fold_id",
    "test_months",
    "gap_months",
    "window_label",
    "sku_segment",
    "forecast_population",
]
available_columns = pd.read_csv(sku_forecast_path, nrows=0).columns.tolist()
use_columns = [column for column in sku_forecast_columns if column in available_columns]
sku_forecasts = pd.read_csv(sku_forecast_path, usecols=use_columns, parse_dates=["month"])

required_sku_forecasts = sku_forecasts.loc[
    sku_forecasts["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & sku_forecasts["gap_months"].eq(REQUIRED_LAG_MONTHS)
    & sku_forecasts["model"].eq(lead_model)
].copy()
sku_fold_column = "global_fold_id" if "global_fold_id" in required_sku_forecasts.columns else "fold_id"
latest_sku_fold = required_sku_forecasts[sku_fold_column].max()
latest_sku_forecasts = required_sku_forecasts.loc[
    required_sku_forecasts[sku_fold_column].eq(latest_sku_fold)
].copy()

latest_sku_forecasts["absolute_error"] = (
    latest_sku_forecasts["demand"] - latest_sku_forecasts["forecast"]
).abs()
sku_example_summary = (
    latest_sku_forecasts.groupby(["sku_id"], as_index=False)
    .agg(
        sku_segment=("sku_segment", "first") if "sku_segment" in latest_sku_forecasts.columns else ("model", "first"),
        actual_horizon=("demand", "sum"),
        forecast_horizon=("forecast", "sum"),
        sku_month_absolute_error=("absolute_error", "sum"),
        active_months=("demand", lambda values: int((values > 0).sum())),
        forecast_positive_months=("forecast", lambda values: int((values > 0).sum())),
    )
)
sku_example_summary["horizon_absolute_error"] = (
    sku_example_summary["actual_horizon"] - sku_example_summary["forecast_horizon"]
).abs()
sku_example_summary["sku_month_wmape_percent"] = np.where(
    sku_example_summary["actual_horizon"].gt(0),
    100 * sku_example_summary["sku_month_absolute_error"] / sku_example_summary["actual_horizon"],
    np.nan,
)
sku_example_summary["sku_horizon_wmape_percent"] = np.where(
    sku_example_summary["actual_horizon"].gt(0),
    100 * sku_example_summary["horizon_absolute_error"] / sku_example_summary["actual_horizon"],
    np.nan,
)

sku_example_summary = (
    sku_example_summary.loc[sku_example_summary["actual_horizon"].gt(0)]
    .sort_values(["actual_horizon", "sku_month_wmape_percent"], ascending=[False, True])
    .head(5)
    .reset_index(drop=True)
)
selected_skus = sku_example_summary["sku_id"].tolist()

sku_example_forecasts = latest_sku_forecasts.loc[
    latest_sku_forecasts["sku_id"].isin(selected_skus)
].copy()
sku_example_forecasts = sku_example_forecasts.merge(
    sku_example_summary[
        [
            "sku_id",
            "actual_horizon",
            "forecast_horizon",
            "sku_month_wmape_percent",
            "sku_horizon_wmape_percent",
        ]
    ],
    on="sku_id",
    how="left",
)
sku_example_forecasts = sku_example_forecasts.sort_values(["actual_horizon", "sku_id", "month"], ascending=[False, True, True])

summary_path = table_dir / "lumpy_sku_forecast_example_summary_latest_required.csv"
forecast_path = table_dir / "lumpy_sku_forecast_examples_latest_required.csv"
sku_example_summary.to_csv(summary_path, index=False)
sku_example_forecasts.to_csv(forecast_path, index=False)

print(f"Latest required fold: {latest_sku_fold}")
print(f"Lead model: {lead_model}")
print(f"Saved SKU example summary: {summary_path}")
print(f"Saved SKU example monthly forecasts: {forecast_path}")

display(
    sku_example_summary[
        [
            "sku_id",
            "sku_segment",
            "actual_horizon",
            "forecast_horizon",
            "sku_month_wmape_percent",
            "sku_horizon_wmape_percent",
            "active_months",
            "forecast_positive_months",
        ]
    ]
)
display(
    sku_example_forecasts[
        [
            "sku_id",
            "month",
            "demand",
            "forecast",
            "sku_month_wmape_percent",
            "sku_horizon_wmape_percent",
        ]
    ]
)

if go is None:
    print("Plotly is not installed, so the SKU-level forecast chart is skipped.")
else:
    fig = go.Figure()
    colors = ["#2563eb", "#16a34a", "#dc2626", "#9333ea", "#f59e0b"]
    for sku_index, sku_id in enumerate(selected_skus):
        sku_data = sku_example_forecasts.loc[sku_example_forecasts["sku_id"].eq(sku_id)].sort_values("month")
        summary_row = sku_example_summary.loc[sku_example_summary["sku_id"].eq(sku_id)].iloc[0]
        label = f"{sku_id} ({summary_row['sku_month_wmape_percent']:.1f}% WMAPE)"
        fig.add_trace(
            go.Scatter(
                x=sku_data["month"],
                y=sku_data["demand"],
                mode="lines+markers",
                name=f"Actual - {label}",
                line=dict(color=colors[sku_index % len(colors)], width=3),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=sku_data["month"],
                y=sku_data["forecast"],
                mode="lines+markers",
                name=f"Forecast - {label}",
                line=dict(color=colors[sku_index % len(colors)], width=2, dash="dash"),
            )
        )

    fig.update_layout(
        title=f"SKU-Level Forecast Examples - {lead_model}, Latest {REQUIRED_WINDOW_MONTHS}m/{REQUIRED_LAG_MONTHS}m Fold",
        xaxis_title="Month",
        yaxis_title="SKU demand",
        hovermode="x unified",
        height=650,
        margin=dict(l=60, r=30, t=80, b=70),
    )
    fig.show()


## Trend Shape Check


In [ ]:
def linear_slope(values):
    values = pd.Series(values).astype(float).to_numpy()
    if len(values) < 2 or np.nanstd(values) == 0:
        return 0.0
    return float(np.polyfit(np.arange(len(values), dtype=float), values, 1)[0])


if monthly_totals.empty:
    print("No monthly-total detail table found, so the trend shape check is skipped.")
else:
    required_monthly = monthly_totals.loc[
        monthly_totals["test_months"].eq(REQUIRED_WINDOW_MONTHS)
        & monthly_totals["gap_months"].eq(REQUIRED_LAG_MONTHS)
    ].copy()
    fold_column = "global_fold_id" if "global_fold_id" in required_monthly.columns else "fold_id"
    latest_fold = required_monthly[fold_column].max()
    latest = required_monthly.loc[required_monthly[fold_column].eq(latest_fold)].copy()

    rows = []
    for model_name, model_data in latest.groupby("model"):
        model_data = model_data.sort_values("month")
        actual_slope = linear_slope(model_data["actual"])
        forecast_slope = linear_slope(model_data["forecast"])
        actual_std = float(model_data["actual"].std(ddof=0))
        forecast_std = float(model_data["forecast"].std(ddof=0))
        corr = (
            float(np.corrcoef(model_data["actual"], model_data["forecast"])[0, 1])
            if actual_std > 0 and forecast_std > 0
            else np.nan
        )
        flat_threshold = max(abs(actual_slope) * 0.25, 1.0)
        if abs(actual_slope) < 1.0:
            trend_status = "actual_is_flat"
        elif abs(forecast_slope) < flat_threshold:
            trend_status = "flat_forecast_vs_actual_trend"
        elif np.sign(actual_slope) != np.sign(forecast_slope):
            trend_status = "wrong_direction"
        else:
            trend_status = "trend_direction_present"

        rows.append(
            {
                "model": model_name,
                "latest_fold": latest_fold,
                "actual_slope_per_month": actual_slope,
                "forecast_slope_per_month": forecast_slope,
                "actual_forecast_correlation": corr,
                "latest_fold_monthly_total_wmape_percent": (
                    100 * (model_data["actual"] - model_data["forecast"]).abs().sum() / model_data["actual"].sum()
                    if model_data["actual"].sum() > 0
                    else np.nan
                ),
                "trend_status": trend_status,
            }
        )

    trend_shape = pd.DataFrame(rows).sort_values(["latest_fold_monthly_total_wmape_percent", "model"])
    display(trend_shape)

    lead_status = trend_shape.loc[trend_shape["model"].eq(lead_model), "trend_status"]
    if not lead_status.empty and lead_status.iloc[0] == "flat_forecast_vs_actual_trend":
        print(
            f"{lead_model} is scoring well on level/rolling WMAPE, but the latest-fold line is flat against a moving actual series."
        )


## SKU-Horizon Check


In [ ]:
horizon_columns = [
    "window_label",
    "model",
    "horizon_sku_wmape_percent",
    "median_positive_sku_wmape_percent",
    "positive_actual_skus",
    "missed_positive_skus",
    "false_positive_zero_actual_sku_share_percent",
]
display(sku_horizon_required.sort_values(["horizon_sku_wmape_percent", "model"])[horizon_columns])

if not lag_comparison_report.empty:
    display(lag_comparison_report.sort_values(["test_months", "best_lag_months"]))

if not metric_decision_report.empty:
    display(
        metric_decision_report.loc[
            metric_decision_report["test_months"].eq(REQUIRED_WINDOW_MONTHS)
            & metric_decision_report["gap_months"].eq(REQUIRED_LAG_MONTHS)
        ]
    )
